                # LabeledFewShot

                Sample labeled training examples and attach them directly as demonstrations; no teacher calls are needed during compilation.

                **Use it when:** You have trustworthy labels, want the cheapest few-shot baseline, and do not need generated reasoning traces.

                **What compilation changes:** Demonstrations only; the original instruction remains unchanged.

                Important configuration in this benchmark:

                - `k=4` caps prompt growth
- `sample=True` samples from the frozen training split

                The 74-row dataset and pair-grouped train/validation/test membership are frozen.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import (
    artifact_paths,
    benchmark_snapshot,
    learned_program_preview,
    verify_prompt_artifact,
)

OPTIMIZER = 'labeled-few-shot'
RUN_MODE = os.getenv("CHAPTER06_RUN", "inspect").lower()
print(f"mode={RUN_MODE!r}; optimizer={OPTIMIZER!r}")
print("safe default: inspect checked-in full-run artifacts; no API calls")

mode='inspect'; optimizer='labeled-few-shot'
safe default: inspect checked-in full-run artifacts; no API calls


                ## Compile shape

                This is the essential DSPy call used by the shared runner (setup variables omitted):

                ```python
                optimizer = dspy.LabeledFewShot(k=4)
optimized_detector = optimizer.compile(detector, trainset=trainset, sample=True)
                ```

                `compile` returns a program. Calling that program on the untouched test examples is
                a separate phase; the notebook reports optimization cost/time separately from inference latency.

In [2]:
print(benchmark_snapshot(OPTIMIZER))
print()
print(artifact_paths(OPTIMIZER))

LabeledFewShot — frozen full-profile rerun
status: completed
task model: openai/gpt-5.6-luna
test accuracy: 75.0%
uplift: +25.0 points vs Luna reference
optimization: $0.0000 and 0.0s
inference latency: mean 2.29s; p95 3.50s
reload checks: prompt=True; model=None; predictions=3/3 labels
complete run: chapter06/results/runs/full/labeled-few-shot/20260715T021433.053902Z

Complete artifacts:
- complete_output: chapter06/results/runs/full/labeled-few-shot/20260715T021433.053902Z/complete_output.txt
- lm_history: chapter06/results/runs/full/labeled-few-shot/20260715T021433.053902Z/lm_history.jsonl
- metrics: chapter06/results/runs/full/labeled-few-shot/20260715T021433.053902Z/metrics.json
- predictions: chapter06/results/runs/full/labeled-few-shot/20260715T021433.053902Z/predictions.jsonl
- program: chapter06/results/runs/full/labeled-few-shot/20260715T021433.053902Z/program.json
- prompts: chapter06/results/runs/full/labeled-few-shot/20260715T021433.053902Z/prompts.json
- canonical program

## Read the result

Inspect the four saved demos. This method can improve in-context calibration, but sampled examples are not selected against the validation set.

The next cell shows a bounded readable preview. The complete, lossless prompt and
serialized program remain at the paths printed above.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 4
1. is_ai=False: join is downstream of follow_branch_a and branch_false. The join task will show up as skipped because its trigger_rule is set to all_success by default, and the skip caused by t…
2. is_ai=False: A given blog article might have various statuses - for instance, it might be visible to everyone (i.e. public), or only visible to the author (i.e. private). It may also be hidd…
3. is_ai=False: You can adjust the configuration of cmake variables optionally (without building first), by doing the following. For example, adjusting the pre-detected directories for CuDNN or…
4. is_ai=False: When we submit the form, the POST /articles request is mapped to the create action. The create action does attempt to save @article. Therefore, validations are checked. If any v…

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt

## Run it yourself (explicit opt-in)

The default `inspect` mode is offline and free. For a live run, first install from
the repository root with `python -m pip install -r requirements.txt`, configure
`OPENAI_API_KEY`, and restart the kernel.

- Bounded code-path check: launch Jupyter with `CHAPTER06_RUN=smoke`.
- Complete frozen-split rerun: launch Jupyter with `CHAPTER06_RUN=full`.

A full run is intentionally not triggered by opening or choosing “Run All”: it can
take minutes or incur model charges. The weight optimizers download and train a
small Qwen model locally through MPS/CPU and require the optional training stack. Live
artifacts are written to `chapter06/results/runs/<profile>/<optimizer>/<run-id>/`.
Rebuild the comparison afterward with `python -m chapter06.summarize_optimizer_results`.

In [4]:
if RUN_MODE == "inspect":
    print("Live run skipped. Set CHAPTER06_RUN=smoke or CHAPTER06_RUN=full explicitly.")
elif RUN_MODE in {"smoke", "full"}:
    from chapter06.optimizer_experiment import run_experiment

    live_result = run_experiment(OPTIMIZER, profile_name=RUN_MODE)
    print(live_result)
else:
    raise ValueError("CHAPTER06_RUN must be inspect, smoke, or full")

Live run skipped. Set CHAPTER06_RUN=smoke or CHAPTER06_RUN=full explicitly.
